# Bank Marketing Classification

This notebook explores the UCI Bank Marketing data, prepares mixed numeric and
categorical features, handles class imbalance, compares five classifiers, and
tunes their hyperparameters. Run it from the `notebooks/` directory after
installing the packages in `requirements.txt`.


In [ ]:
import pandas as pd        
import numpy as np           
import matplotlib.pyplot as plt

In [ ]:
from pathlib import Path

DATA_PATH = Path("../data/bank_marketing.csv")
bank_df = pd.read_csv(DATA_PATH, sep=";")
print(f"Loaded {len(bank_df):,} rows from {DATA_PATH}")


In [ ]:
bank_df.head()

In [ ]:
bank_df.describe()

In [ ]:
bank_df.info()

In [ ]:
print("Shape:", bank_df.shape)

print("\nColumn names:")
print(bank_df.columns)

print("\nData types:")
print(bank_df.dtypes)




In [ ]:
print("\nMissing values per column:")
print(bank_df.isnull().sum())

In [ ]:
print(bank_df["y"].value_counts())
bank_df["y"].value_counts().plot(kind="bar", color=["skyblue", "salmon"])
plt.title("Class Distribution (Target Variable: y)")
plt.xlabel("Class")
plt.ylabel("Count")
plt.show()



In [ ]:
bank_df["y"].value_counts(normalize=True) * 100


In [ ]:
# Numerical columns (int and float)
numeric_cols = bank_df.select_dtypes(include=["int64", "float64"]).columns

# Categorical columns (object or string)
categorical_cols = bank_df.select_dtypes(include=["object", "category"]).columns

print("Numeric columns:", list(numeric_cols))
print("Categorical columns:", list(categorical_cols))




In [ ]:
for col in numeric_cols:
    plt.hist(bank_df[col], bins=30, color='steelblue', edgecolor='black')
    plt.title(f"Distribution of {col}")
    plt.xlabel(col)
    plt.ylabel("Frequency")
    plt.show()


In [ ]:
for col in numeric_cols:
    plt.figure(figsize=(6,4))
    plt.boxplot(bank_df[col].dropna())
    plt.title(f"Boxplot of {col}")
    plt.ylabel(col)
    plt.show()


In [ ]:
# Categorical columns
categorical_cols = bank_df.select_dtypes(include=["object"]).columns

for col in categorical_cols:
    plt.figure(figsize=(8,4))
    bank_df[col].value_counts().plot(kind="bar", color="orange")
    plt.title(f"Bar Plot of {col}")
    plt.xlabel("Categories")
    plt.ylabel("Count")
    plt.show()


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

for col in numeric_cols:
    plt.figure(figsize=(6,4))
    sns.boxplot(x='y', y=col, data=bank_df)
    plt.title(f"{col} vs Target (y)")
    plt.show()


In [ ]:
for col in categorical_cols:
    print(f"\n\nFeature: {col}")
    print(pd.crosstab(bank_df[col], bank_df['y'], normalize='index') * 100)


In [ ]:
for col in categorical_cols:
    plt.figure(figsize=(8,4))
    pd.crosstab(bank_df[col], bank_df['y']).plot(kind='bar', stacked=True)
    plt.title(f"Relationship between {col} and y")
    plt.xlabel(col)
    plt.ylabel("Count")
    plt.show()


In [ ]:
plt.figure(figsize=(12,8))
sns.heatmap(bank_df[numeric_cols].corr(),  annot=True, cmap="coolwarm", fmt='.2f')
plt.title("Correlation Heatmap — Numerical Features")
plt.show()



In [ ]:
# Correlation screening is learned from the training partition later.
# This full-data matrix is used only for exploratory reporting.
corr_matrix = bank_df[numeric_cols].corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
eda_high_correlation = [col for col in upper.columns if any(upper[col] > 0.90)]
print("Highly correlated columns (EDA only):", eda_high_correlation)


In [ ]:
bank_df.head()

In [ ]:
from sklearn.ensemble import IsolationForest

# Exploratory outlier labels. The model-training filter is fitted only on the
# training partition after the split to avoid test-set leakage.
numeric_cols = bank_df.select_dtypes(include=["int64", "float64"]).columns
iso_eda = IsolationForest(contamination=0.05, random_state=42)
iso_eda.fit(bank_df[numeric_cols])


In [ ]:
# Exploratory predictions: 1 = normal, -1 = outlier
iso_labels = iso_eda.predict(bank_df[numeric_cols])
bank_df["is_outlier_iso"] = np.where(iso_labels == -1, 1, 0)
bank_df[["is_outlier_iso"]].head()


In [ ]:
bank_df['is_outlier_iso'].value_counts()


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

for col in numeric_cols:
    plt.figure(figsize=(7,4))
    sns.scatterplot(
        x=bank_df.index,
        y=bank_df[col],
        hue=bank_df['is_outlier_iso'],
        palette={0:'blue', 1:'red'},
        alpha=0.6
    )
    plt.title(f"Isolation Forest Outliers on {col}")
    plt.xlabel("Index")
    plt.ylabel(col)
    plt.show()


In [ ]:
# Keep the full dataset for splitting. Training-only outlier filtering
# is applied below; the held-out test set remains untouched.
bank_df_model = bank_df.drop(columns=["is_outlier_iso"]).copy()


Separate features (X) and target (y)

In [ ]:
# Separate features (X) and target (y)
X = bank_df_model.drop(columns=["y"])
y = bank_df_model["y"]


In [ ]:
X.head()

# test and train split

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import IsolationForest

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y,
)

# Learn correlation filtering from training data only.
train_numeric_cols = X_train.select_dtypes(include=["int64", "float64"]).columns
train_corr = X_train[train_numeric_cols].corr().abs()
train_upper = train_corr.where(
    np.triu(np.ones(train_corr.shape), k=1).astype(bool)
)
to_drop = [col for col in train_upper.columns if any(train_upper[col] > 0.90)]
X_train = X_train.drop(columns=to_drop).copy()
X_test = X_test.drop(columns=to_drop).copy()

# Fit the outlier detector on training rows only and never filter the test set.
train_numeric_cols = X_train.select_dtypes(include=["int64", "float64"]).columns
iso_train = IsolationForest(contamination=0.05, random_state=42)
normal_train_mask = iso_train.fit_predict(X_train[train_numeric_cols]) == 1
X_train = X_train.loc[normal_train_mask].copy()
y_train = y_train.loc[normal_train_mask].copy()

print("Dropped correlated columns:", to_drop)
print("Train/test shapes:", X_train.shape, X_test.shape)


# encode

In [ ]:
y_train = y_train.map({'no': 0, 'yes': 1})
y_test  = y_test.map({'no': 0, 'yes': 1})


In [ ]:
ordinal_cols = ['month', 'day_of_week', 'education']

ordinal_categories = [
    ['jan','feb','mar','apr','may','jun','jul','aug','sep','oct','nov','dec'],
    ['mon','tue','wed','thu','fri'],
    ['illiterate','basic.4y','basic.6y','basic.9y',
     'high.school','professional.course','university.degree','unknown']
]

# Nominal columns (for frequency encoding)
nominal_cols = ['job','marital','default','housing','loan','contact','poutcome']


In [ ]:
from sklearn.preprocessing import OrdinalEncoder

ord_enc = OrdinalEncoder(categories=ordinal_categories)
X_train[ordinal_cols] = ord_enc.fit_transform(X_train[ordinal_cols])
X_test[ordinal_cols]  = ord_enc.transform(X_test[ordinal_cols])


In [ ]:
for col in nominal_cols:
    freq = X_train[col].value_counts(normalize=True)
    X_train[col] = X_train[col].map(freq).fillna(0.0)
    X_test[col] = X_test[col].map(freq).fillna(0.0)


In [ ]:
X_train.head()

# scale

In [ ]:
from sklearn.preprocessing import StandardScaler
import pandas as pd

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)


In [ ]:
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline

sm = SMOTE(k_neighbors=5, random_state=42)
X_train_resampled, y_train_resampled = sm.fit_resample(X_train_scaled, y_train)

print("Before SMOTE:", y_train.value_counts())
print("After SMOTE:", y_train_resampled.value_counts())


In [ ]:
plt.bar(y_train_resampled.value_counts().index, y_train_resampled.value_counts().values, color="green")
plt.title("Class Distribution After SMOTE")
plt.xlabel("Class")
plt.ylabel("Count")
plt.show()


In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report


In [ ]:
from sklearn.metrics import classification_report
def evaluate_model(model, X_train, y_train, X_test, y_test, name):
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    
    print(f"\n===== {name} =====")
    print("Accuracy :", accuracy_score(y_test, y_pred))
    
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, zero_division=0))
    
    print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))


In [ ]:
# Baseline models evaluated on the same held-out test partition.
dt = DecisionTreeClassifier(random_state=42)
svm = SVC(kernel="linear", random_state=42)
logreg = LogisticRegression(max_iter=500, random_state=42)
knn = KNeighborsClassifier(n_neighbors=5)
xgb = XGBClassifier(
    random_state=42,
    eval_metric="logloss",
    tree_method="hist",
)


In [ ]:
evaluate_model(dt,    X_train_resampled, y_train_resampled, X_test_scaled, y_test, "Decision Tree")
evaluate_model(svm,   X_train_resampled, y_train_resampled, X_test_scaled, y_test, "SVM (Linear)")
evaluate_model(logreg,X_train_resampled, y_train_resampled, X_test_scaled, y_test, "Logistic Regression")
evaluate_model(knn,   X_train_resampled, y_train_resampled, X_test_scaled, y_test, "KNN (k=5)")
evaluate_model(xgb,   X_train_resampled, y_train_resampled, X_test_scaled, y_test, "XGBoost")


In [ ]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    RocCurveDisplay,
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


def smote_pipeline(estimator):
    # SMOTE runs inside each CV training fold, preventing synthetic samples
    # derived from a validation fold from leaking into that fold's training set.
    return Pipeline([
        ("smote", SMOTE(k_neighbors=5, random_state=42)),
        ("model", estimator),
    ])


In [ ]:
def evaluate_model_trained(model, X_test, y_test, name):
    y_pred = model.predict(X_test)
    
    acc  = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec  = recall_score(y_test, y_pred)
    f1   = f1_score(y_test, y_pred)
    
    print(f"\n===== {name} =====")
    print("Accuracy :", acc)
    print("Precision:", prec)
    print("Recall   :", rec)
    print("F1-score :", f1)
    print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
    
    return acc, prec, rec, f1


In [ ]:
dt_params = {
    "model__criterion": ["gini", "entropy"],
    "model__max_depth": [3, 5, 7, 9, None],
}

dt_grid = GridSearchCV(
    smote_pipeline(DecisionTreeClassifier(random_state=42)),
    dt_params,
    cv=cv,
    scoring="f1",
    n_jobs=2,
)
dt_grid.fit(X_train_scaled, y_train)
best_dt = dt_grid.best_estimator_
print("Best Decision Tree params:", dt_grid.best_params_)


In [ ]:
logreg_params = {
    "model__penalty": ["l1", "l2"],
    "model__C": [0.01, 0.1, 1, 10],
    "model__solver": ["liblinear"],
}

logreg_grid = GridSearchCV(
    smote_pipeline(LogisticRegression(max_iter=500, random_state=42)),
    logreg_params,
    cv=cv,
    scoring="f1",
    n_jobs=2,
)
logreg_grid.fit(X_train_scaled, y_train)
best_logreg = logreg_grid.best_estimator_
print("Best Logistic Regression params:", logreg_grid.best_params_)


In [ ]:
knn_params = {
    "model__n_neighbors": [3, 5, 7, 9, 11],
    "model__metric": ["euclidean", "manhattan"],
}

knn_grid = GridSearchCV(
    smote_pipeline(KNeighborsClassifier()),
    knn_params,
    cv=cv,
    scoring="f1",
    n_jobs=2,
)
knn_grid.fit(X_train_scaled, y_train)
best_knn = knn_grid.best_estimator_
print("Best KNN params:", knn_grid.best_params_)


In [ ]:
svm_params = {
    "model__kernel": ["linear", "rbf", "poly"],
    "model__C": [0.1, 1, 10],
}

svm_grid = GridSearchCV(
    smote_pipeline(SVC(random_state=42)),
    svm_params,
    cv=cv,
    scoring="f1",
    n_jobs=2,
)
svm_grid.fit(X_train_scaled, y_train)
best_svm = svm_grid.best_estimator_
print("Best SVM params:", svm_grid.best_params_)


In [ ]:
xgb_params = {
    "model__n_estimators": [100, 200],
    "model__max_depth": [3, 5, 7],
    "model__learning_rate": [0.01, 0.1, 0.2],
}

xgb_grid = GridSearchCV(
    smote_pipeline(
        XGBClassifier(
            random_state=42,
            eval_metric="logloss",
            tree_method="hist",
        )
    ),
    xgb_params,
    cv=cv,
    scoring="f1",
    n_jobs=2,
)
xgb_grid.fit(X_train_scaled, y_train)
best_xgb = xgb_grid.best_estimator_
print("Best XGBoost params:", xgb_grid.best_params_)


In [ ]:
import time

models = {
    "Decision Tree": best_dt,
    "Logistic Regression": best_logreg,
    "KNN": best_knn,
    "SVM": best_svm,
    "XGBoost": best_xgb
}

results = []

for name, model in models.items():
    start = time.time()
    model.fit(X_train_scaled, y_train)
    train_time = time.time() - start
    
    acc, prec, rec, f1 = evaluate_model_trained(model, X_test_scaled, y_test, name)
    
    results.append({
        'Model': name,
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1 Score': f1,
        'Train Time (sec)': train_time
    })

results_df = pd.DataFrame(results)
print("\n\n=== Summary Results ===")
print(results_df)


In [ ]:
for name, model in models.items():
    disp = ConfusionMatrixDisplay.from_estimator(model, X_test_scaled, y_test)
    plt.title(f"Confusion Matrix - {name}")
    plt.show()


In [ ]:
for name, model in models.items():
    RocCurveDisplay.from_estimator(model, X_test_scaled, y_test)
    plt.title(f"ROC Curve - {name}")
    plt.show()
